# 1. Gradio入门
> https://gradio.org.cn/

Gradio 是一个非常方便的 Python 库，用于快速搭建机器学习模型或函数的 Web 可视化界面。你只需要几行代码，就可以把一个 Python 函数转换成交互式的网页应用。它常用于：
- 模型 Demo 展示；
- 快速调试模型输入输出；
- 与非技术人员分享模型功能；
- 搭配 Hugging Face Spaces 发布应用。

安装： `pip install gradio`
## 1. Gradio 的基本用法

Gradio 的核心是 `gr.Interface`：

In [1]:
import gradio as gr

def 函数名(输入参数):
    return 输出结果

demo = gr.Interface(fn=函数名, inputs=输入组件, outputs=输出组件)
demo.launch()

# 1. fn：要包装的 Python 函数；
# 2. inputs：定义输入控件（如文本框、滑块、图片上传等）；
# 3. outputs：定义输出控件（如文本、图片、标记等）；
# 4. launch()：启动本地或在线的应用。

ModuleNotFoundError: No module named 'gradio'

## 2. 简单示例：文本情感判断
> 下面用 Python + Gradio 写一个简单例子，实现一个非常简单的情感识别（非模型，只是示范逻辑）：

In [1]:
import gradio as gr

# 定义功能函数
def sentiment_analysis(text):
    text = text.lower()
    if "好" in text or "开心" in text or "喜欢" in text:
        return "积极 😊"
    elif "不好" in text or "难过" in text or "讨厌" in text:
        return "消极 😞"
    else:
        return "中性 😐"

# 创建 Gradio 界面
demo = gr.Interface(
    fn=sentiment_analysis,                 # 处理函数
    inputs=gr.Textbox(label="请输入文本"),  # 输入组件为文本框
    outputs=gr.Textbox(label="情感分析结果"),# 输出组件
    title="情感分析示例",
    description="输入一句话，判断它的情感倾向（简单示例）"
)

# 启动应用
demo.launch(share=True)


/usr/local/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d4c50a4b6764133409.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


运行后，会自动在浏览器中打开一个简单的界面：

- 输入一句话，例如 “我今天很开心！”
- 点击 “提交” 按钮；
- 输出框中显示结果 “积极 😊”。

## 3. 更多组件
gr.Image(), gr.Audio(), gr.Video(), gr.Slider(), gr.Dropdown(), ...

# 2. Gradio - 构建应用

## 1. 文本分类 - 情感分析

In [ ]:
# app.py
import gradio as gr
from transformers import pipeline

# 载入 Hugging Face 模型（情感分析）
sentiment = pipeline("sentiment-analysis")

def analyze_text(text):
    result = sentiment(text)[0]
    return f"标签: {result['label']}，置信度: {result['score']:.2f}"

demo = gr.Interface(
    fn=analyze_text,
    inputs=gr.Textbox(lines=3, placeholder="输入一句话"),
    outputs="text",
    title="情感分析",
    description="使用 Hugging Face Transformers 进行情感分析"
)

demo.launch(share=True)


## 2. 计算机视觉 — 目标检测

In [ ]:
# app.py
import gradio as gr
from transformers import pipeline

# 使用 Hugging Face 的目标检测模型
detector = pipeline("object-detection", model="facebook/detr-resnet-50", revision="no_timm")

def detect_objects(image):
    results = detector(image)
    return results

demo = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(type="filepath", label="上传图片"),
    outputs=gr.JSON(label="检测结果"),
    title="图片目标检测",
    description="使用 DETR 模型检测图片中的物体"
)

demo.launch(share=True)


扩展：gradio 展示输出的图片。
1. 函数返回图片
2. 输出控件为图片
> 
    # ---- 关键：将 Matplotlib 图像对象转成 PIL.Image ----
    # import matplotlib.pyplot as plt
    # import io
    # from PIL import Image
    # buf = io.BytesIO()
    # fig.savefig(buf, format='png')  # 保存到内存中
    # buf.seek(0)
    # img = Image.open(buf)  # 转成 PIL 图像
    # plt.close(fig)         # 释放 Matplotlib 资源

In [ ]:
# app.py
import gradio as gr
from transformers import pipeline
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import io
import numpy as np

# 使用 Hugging Face 的目标检测模型
detector = pipeline("object-detection", model="facebook/detr-resnet-50", revision="no_timm")

def detect_objects(image_path):
    # 读取图片
    image = Image.open(image_path)
    
    # 进行目标检测
    results = detector(image)
    
    # 绘制结果
    img = draw_img(image, results)
    return img

# 绘图方法
def draw_img(image, results):
    # 打印检测结果
    for r in results:
        print(f"{r['label']} {r['score']:.3f} {r['box']}")
    
    # 创建图形
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    
    # 画框和标签
    for r in results:
        box = r["box"]
        x, y, w, h = box["xmin"], box["ymin"], box["xmax"] - box["xmin"], box["ymax"] - box["ymin"]
        
        # 绘制矩形框
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        # 添加标签
        ax.text(x, y-5, f"{r['label']} {r['score']:.3f}", 
                fontsize=10, color='white', 
                bbox=dict(facecolor='red', alpha=0.7, pad=2))
    
    ax.axis("off")
    plt.tight_layout()
    
    # 保存到内存
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=150, pad_inches=0)
    buf.seek(0)
    
    # 转换为PIL图像
    result_image = Image.open(buf)
    
    # 关闭matplotlib图形以释放内存
    plt.close(fig)
    
    return result_image

# 创建Gradio界面
demo = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(type="filepath", label="上传图片"),
    outputs=gr.Image(type="pil", label="检测结果"),
    title="图片目标检测",
    description="使用 DETR 模型检测图片中的物体",
    examples=[],  # 你可以在这里添加示例图片路径
    cache_examples=False
)

demo.launch(share=True)


## 3. 更多用法
> AI提示词：写一个 Qwen/Qwen-Image 大模型的文生图 gradio 应用页面，要求可以选择清晰度等